# 01 — Data Cleaning
**Week 2, Steps 1–6**

Loads all 12 raw CSVs, applies the `DataCleaner` pipeline, shows before/after
profiles for each table, and displays the cleaning log.

> Run **Restart & Run All** to reproduce from scratch.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 4)

from src.ingest import load_all
from src.clean import CleaningLog, DataCleaner, CITY_ALIASES, iqr_bounds

rng = np.random.default_rng(42)  # seeded RNG for any sampling (PRD requirement)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_DIR   = Path('data/raw')
CLEAN_DIR = Path('data/cleaned')
print("Libraries loaded ✅")


## 1. Load raw CSVs

In [ ]:
raw = load_all()
print(f"Tables loaded: {len(raw)}")
summary = pd.DataFrame([
    {"table": name, "rows": len(df), "cols": len(df.columns)}
    for name, df in sorted(raw.items())
])
summary


## 2. Run cleaning pipeline

In [ ]:
cities_list = raw['cities']['city'].dropna().str.strip().tolist()
log = CleaningLog()
cleaner = DataCleaner(cities_list, CITY_ALIASES, log)
cleaned = cleaner.clean_all(raw)
print(f"Cleaning complete. Log entries: {len(log.rows)}")


## 3. Before / After row counts

In [ ]:
comparison = pd.DataFrame([
    {"table": t, "raw_rows": len(raw[t]), "cleaned_rows": len(cleaned[t]),
     "removed": len(raw[t]) - len(cleaned[t])}
    for t in sorted(cleaned)
])
comparison["removal_%"] = (comparison["removed"] / comparison["raw_rows"] * 100).round(2)
comparison


## 4. Cleaning log

In [ ]:
log_df = pd.DataFrame(log.rows)
log_df.sort_values(["table","rows_affected"], ascending=[True, False])


## 5. City canonical check

In [ ]:
city_tables = ['customers', 'restaurants', 'delivery_partners', 'weather', 'traffic']
for tbl in city_tables:
    unknown = set(cleaned[tbl]['city'].dropna()) - set(cities_list)
    status = "✅ all canonical" if not unknown else f"❌ {sorted(unknown)}"
    print(f"{tbl:<22} {status}")


## 6. Category values match SQL

In [ ]:
checks = {
    "orders.order_status":               ["Cancelled","Delivered","Delivered Late","Food Not Delivered"],
    "orders.payment_method":             ["Cash On Delivery","Credit Card","Debit Card","Net Banking","Upi","Wallet"],
    "customers.membership":              ["Basic","Gold","Zomato Pro"],
    "customers.gender":                  ["Female","Male","Other"],
    "customer_feedback.sentiment":       ["Negative","Neutral","Positive"],
    "delivery_partners.vehicle_type":    ["Bicycle","Bike","Car","Scooter"],
    "payments.payment_status":           ["Failed","Pending","Refunded","Success"],
    "menu.availability":                 ["No","Yes"],
}
for col_path, expected in checks.items():
    tbl, col = col_path.split(".")
    actual = sorted(cleaned[tbl][col].dropna().unique().tolist())
    ok = actual == expected
    print(f"{'✅' if ok else '❌'} {col_path}: {actual}")


## 7. Missing value strategies

In [ ]:
strategies = {
    "customers.age":                 "Impossible → NaN, filled with membership median, age_was_missing flag",
    "delivery_partners.age":         "Impossible → NaN, filled with overall median, age_was_missing flag",
    "restaurants.rating":            "Out-of-range → NaN, filled with city median",
    "delivery_partners.rating":      "Out-of-range → NaN, filled with city median",
    "customer_feedback.*_rating":    "LEFT MISSING — missing = didn't rate",
    "customers.phone":               "Invalid → NaN, 133 repaired, phone_valid flag",
    "customers.email":               "Invalid → NaN, email_valid flag",
    "customers.pincode":             "Invalid → NaN, pincode_valid flag",
    "customer_feedback.review":      "LEFT MISSING, has_review flag added",
    "orders.delivery_time_minutes":  "NOT IMPUTED — prediction target; <=0 → NaN",
    "orders.delivery_fee=0":         "KEPT — may be legitimate free delivery",
}
for col, strategy in strategies.items():
    print(f"  {col:<40} {strategy}")


## 8. Outlier analysis — delivery_time_minutes

In [ ]:
dt = cleaned['orders']['delivery_time_minutes'].dropna()

# Percentile table
pcts = np.percentile(dt, [50, 75, 90, 95, 99, 99.9])
pct_df = pd.DataFrame({
    'percentile': ['p50','p75','p90','p95','p99','p99.9'],
    'minutes': pcts.round(1)
})
print("Delivery time percentiles (valid orders, >0):")
print(pct_df.to_string(index=False))

low, high = iqr_bounds(dt, k=3.0)
print(f"\nIQR bounds (k=3.0): [{low:.1f}, {high:.1f}]")
print(f"Flagged outliers: {((dt < low) | (dt > high)).sum()}")
print(f"Bounds decision: use dt_outlier_flag column; Week 3 decides exclusion threshold")

# Before/After histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cleaned['orders']['delivery_time_minutes'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Delivery Time — All valid (>0 min)')
axes[0].set_xlabel('minutes')
axes[1].hist(dt[dt.between(low, high)], bins=40, color='green', edgecolor='white')
axes[1].set_title('Delivery Time — Excluding flagged outliers')
axes[1].set_xlabel('minutes')
plt.tight_layout()
plt.savefig('reports/delivery_time_outliers.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved → reports/delivery_time_outliers.png")


## 9. Cross-tab: delivery time null vs order status

In [ ]:
ct = pd.crosstab(
    cleaned['orders']['order_status'],
    cleaned['orders']['delivery_time_minutes'].isna().map({True:'null',False:'has value'}),
    margins=True
)
print("Finding: most null delivery times are on non-Delivered orders (expected):")
ct


## 10. Cleaning log summary by table

In [ ]:
log_df = pd.DataFrame(log.rows)
pivot = log_df.pivot_table(
    index='table', columns='step', values='rows_affected',
    aggfunc='sum', fill_value=0
)
pivot
